## Quiz 3 Content

In [3]:
import pandas as pd
import numpy as np

In [5]:
circuits = pd.read_csv("f1_data_raw/circuits.csv")
print(circuits.head())

   circuitId   circuitRef                            name      location  \
0          1  albert_park  Albert Park Grand Prix Circuit     Melbourne   
1          2       sepang    Sepang International Circuit  Kuala Lumpur   
2          3      bahrain   Bahrain International Circuit        Sakhir   
3          4    catalunya  Circuit de Barcelona-Catalunya      Montmeló   
4          5     istanbul                   Istanbul Park      Istanbul   

     country       lat        lng  alt  \
0  Australia -37.84970  144.96800   10   
1   Malaysia   2.76083  101.73800   18   
2    Bahrain  26.03250   50.51060    7   
3      Spain  41.57000    2.26111  109   
4     Turkey  40.95170   29.40500  130   

                                                 url  
0  http://en.wikipedia.org/wiki/Melbourne_Grand_P...  
1  http://en.wikipedia.org/wiki/Sepang_Internatio...  
2  http://en.wikipedia.org/wiki/Bahrain_Internati...  
3  http://en.wikipedia.org/wiki/Circuit_de_Barcel...  
4         http://en.w

### Lecture 11 – Subsetting Data

In [6]:
subset_example = circuits.query("lat > 0 and country == 'USA'")
print(subset_example.head())

    circuitId    circuitRef                         name      location  \
18         19  indianapolis  Indianapolis Motor Speedway  Indianapolis   
31         33       phoenix       Phoenix street circuit       Phoenix   
35         37       detroit       Detroit Street Circuit       Detroit   
40         42        dallas                    Fair Park        Dallas   
41         43    long_beach                   Long Beach    California   

   country      lat       lng  alt  \
18     USA  39.7950  -86.2347  223   
31     USA  33.4479 -112.0750  345   
35     USA  42.3298  -83.0401  177   
40     USA  32.7774  -96.7587  139   
41     USA  33.7651 -118.1890   12   

                                                  url  
18  http://en.wikipedia.org/wiki/Indianapolis_Moto...  
31  http://en.wikipedia.org/wiki/Phoenix_street_ci...  
35  http://en.wikipedia.org/wiki/Detroit_street_ci...  
40             http://en.wikipedia.org/wiki/Fair_Park  
41  http://en.wikipedia.org/wiki/Long_Beach,_C

### Lecture 14: Renaming and Replacing

In [7]:
# Rename one column (leaving others unchanged)
circuits = circuits.rename(columns={"name": "circuit_name"})
print(circuits.columns)

# Replace specific values in one line using .replace()
circuits["country"] = circuits["country"].replace({
    "USA": "United States",
    "UK": "United Kingdom",
    "UAE": "United Arab Emirates"
})
print(circuits["country"].unique())

Index(['circuitId', 'circuitRef', 'circuit_name', 'location', 'country', 'lat',
       'lng', 'alt', 'url'],
      dtype='object')
['Australia' 'Malaysia' 'Bahrain' 'Spain' 'Turkey' 'Monaco' 'Canada'
 'France' 'United Kingdom' 'Germany' 'Hungary' 'Belgium' 'Italy'
 'Singapore' 'Japan' 'China' 'Brazil' 'United States'
 'United Arab Emirates' 'Argentina' 'Portugal' 'South Africa' 'Mexico'
 'Korea' 'Netherlands' 'Sweden' 'Austria' 'Morocco' 'Switzerland' 'India'
 'Russia' 'Azerbaijan' 'Saudi Arabia' 'Qatar']


### Lecture 15: Grouping and Aggregating

In [9]:
# Example dataset (simulate results)
df_results = pd.DataFrame({
    "raceId": [1, 1, 2, 2, 3, 3],
    "constructorId": [10, 20, 10, 20, 10, 20],
    "points": [10, 8, 6, 9, 12, 5]
})

# Group and aggregate multiple statistics
df_grouped = (
    df_results.groupby("constructorId")
    .agg(
        mean_points=("points", "mean"),
        max_points=("points", "max"),
        min_points=("points", "min"),
        sd_points=("points", "std")
    )
)
print(df_grouped)

# Subset, group, and aggregate in one line
subset_group_agg = (
    df_results.query("raceId >= 2")
    .groupby(["raceId", "constructorId"])
    .agg(mean_points=("points", "mean"), max_points=("points", "max"))
)
print(subset_group_agg)

               mean_points  max_points  min_points  sd_points
constructorId                                                
10                9.333333          12           6   3.055050
20                7.333333           9           5   2.081666
                      mean_points  max_points
raceId constructorId                         
2      10                     6.0           6
       20                     9.0           9
3      10                    12.0          12
       20                     5.0           5


### Lecture 16: Merging

In [10]:
# Sort by aggregated value in descending order
df_constructor_points_agg = (
    df_results.groupby("constructorId")["points"]
    .agg(avgpoints=("mean"))
    .sort_values(by="avgpoints", ascending=False)
)
print(df_constructor_points_agg)

# Example merge operation
df_races_new = pd.DataFrame({
    "raceId": [1, 2, 3],
    "year": [2020, 2021, 2022],
    "circuitId": [101, 102, 103]
})

df_circuits_new = circuits.rename(columns={"circuitId": "circuitId", "circuit_name": "circuit_name"})

merged = pd.merge(
    left=df_races_new,
    right=df_circuits_new[["circuitId", "circuit_name", "location"]],
    on="circuitId",
    how="left"
)
print(merged.head())

               avgpoints
constructorId           
10              9.333333
20              7.333333
   raceId  year  circuitId circuit_name location
0       1  2020        101          NaN      NaN
1       2  2021        102          NaN      NaN
2       3  2022        103          NaN      NaN
